# ImageNet-R: fair train-only SOHO optimization

This notebook selects SOHO hyperparameters using training data only and compares the locked selection with original FLY fidelity on an untouched outer validation split. Seeds are paired, preregistered replicate factors; they are never selected by accuracy. No held-out test feature is created or read.


In [ ]:
# === Edit repository/path values only. Do not edit grids, seeds or gates. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'experiment/soho-selfcontained'
WORK_DIR = '/content/SOHO-CL'
FEATURE_CACHE_DIR = '/content/soho_imagenetr_gcv_train_cache'
OUTPUT_DIR = '/content/soho_imagenetr_optimal_outputs'
BATCH_SIZE = 128
NUM_WORKERS = 2
EXPECTED_PROTOCOL_SHA256 = 'b1a5b2a819a30c35355540ca3ee3a9f96e3ad5e7672c27d444e8eac192f43004'
EXPECTED_RUNNER_SHA256 = 'f333d010a4345744a12b8ea78a3bffc3abefcc0ab75dee9de403e13229cbf365'


In [ ]:
# Fresh clone, dependencies, GPU check and immutable source verification.
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
os.chdir('/content')
repo = Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','pandas','matplotlib'], check=True)
import torch
assert torch.cuda.is_available(), 'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
PROTOCOL = 'configs/soho_imagenetr_optimal_train_only.json'
RUNNER = 'tools/soho_imagenetr_optimal_train_only.py'
assert sha(PROTOCOL) == EXPECTED_PROTOCOL_SHA256, 'Protocol hash mismatch.'
assert sha(RUNNER) == EXPECTED_RUNNER_SHA256, 'Runner hash mismatch.'
assert not subprocess.check_output(['git','status','--porcelain'], text=True).strip(), 'Repository must start clean.'
print('GPU:', torch.cuda.get_device_name(0))
print('commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('SOURCE CHECK: PASS')


In [ ]:
# Download the verified ViT checkpoint and processed ImageNet-R artifact.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size == 346284714
assert sha(CHECKPOINT_PATH) == '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
DATASET_ROOT = kagglehub.dataset_download('zaphat206/imagenet-r')
print('checkpoint:', CHECKPOINT_PATH)
print('ImageNet-R root:', DATASET_ROOT)


In [ ]:
# Verify exact dataset identity. The expected return code 2 discloses the locked legacy overlap.
AUDIT_PATH = '/content/imagenetr_soho_optimal_audit.json'
audit_run = subprocess.run([sys.executable,'-u','tools/imagenetr_dataset_audit.py','--root',DATASET_ROOT,'--output',AUDIT_PATH,'--expected-identity-sha256','3f3d963b2b0c245ceabc0166c8b1c64d624c2ea31df07ee6ffdbf4cab5f7739d','--diagnose-cross-split-duplicates','--workers','4'])
assert audit_run.returncode == 2, 'Expected locked legacy-overlap disclosure.'
audit = json.loads(Path(AUDIT_PATH).read_text())
assert audit['cross_split_duplicate_content_count'] == 19
assert audit['cross_split_conflicting_label_duplicate_count'] == 18
print('DATASET AUDIT: PASS')


In [ ]:
# Synthetic correctness gate with complete output.
command = [sys.executable,'-m','pytest','-q','tests/test_soho_imagenetr_optimal_train_only.py','tests/test_soho_imagenetr_gcv_diagnostic.py','tests/test_cached_replay_baselines.py','tests/test_soho_selfcontained.py']
completed = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(completed.stdout, flush=True)
if completed.returncode != 0: raise RuntimeError(f'Correctness gate failed: {completed.returncode}')
print('OPTIMAL-SELECTION CORRECTNESS GATE: PASS')


In [ ]:
# Reuse the previous local train cache when available; otherwise extract TRAIN only.
cache = Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command = [sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',DATASET_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256','32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b','--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_imagenetr_optimal','--dataset','ImageNet-R','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','200','--num-tasks','20','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TRAIN FEATURE EXTRACTION START - live task progress follows', flush=True)
    subprocess.run(command, check=True)
else:
    print('Using existing local TRAIN cache:', FEATURE_CACHE_DIR)
assert (cache/'train.pt').is_file() and (cache/'metadata.json').is_file()
assert not (cache/'test.pt').exists(), 'FAIL: test.pt must remain absent.'
metadata = json.loads((cache/'metadata.json').read_text())
assert metadata['train_shape'] == [23918,768] and metadata['test_shape'] is None
print('TRAIN CACHE PASS:', metadata['train_shape'], '| test.pt absent')


In [ ]:
# Locked selection. Safe to rerun after interruption while this runtime/storage survives.
protocol = json.loads(Path(PROTOCOL).read_text())
print('Development seeds:', protocol['selection']['development_replicates'])
print('Outer paired seeds:', protocol['outer_confirmation']['replicates'])
print('Work: 4 ridge configs, 9 representation configs, then 5 paired outer controls.', flush=True)
command = [sys.executable,'-u',RUNNER,'--protocol',PROTOCOL,'--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--dataset-audit',AUDIT_PATH,'--device','cuda']
started = time.time()
completed = subprocess.run(command)
assert completed.returncode == 0, 'Selection failed; return the complete traceback without changing grids or seeds.'
assert Path(OUTPUT_DIR,'selection_results.json').is_file()
assert not Path(FEATURE_CACHE_DIR,'test.pt').exists()
print(f'TRAIN-ONLY SELECTION COMPLETE in {(time.time()-started)/60:.1f} minutes')


In [ ]:
# Show the selected configuration, candidate landscape and paired outer evidence.
import pandas as pd
import matplotlib.pyplot as plt
result = json.loads(Path(OUTPUT_DIR,'selection_results.json').read_text())
print('STATUS:', result['status'])
print('SELECTED SOHO:', result['selected_soho_config'])
print('FLY FIDELITY:', result['fly_fidelity_config'])
print('GATES:', json.dumps(result['gates'], indent=2))
rows = []
for phase, items in [('ridge',result['ridge_results']),('representation',result['representation_results'])]:
    for item in items: rows.append({'phase':phase,**item['config'],'inner_AIA':item['mean_inner_aia']})
candidates = pd.DataFrame(rows); display(candidates.sort_values(['phase','inner_AIA'],ascending=[True,False]))
outer = pd.DataFrame([{'class_seed':x['replicate']['class_order_seed'],'SOHO_AIA':x['soho']['average_incremental_accuracy'],'FLY_AIA':x['fly']['average_incremental_accuracy'],'delta':x['aia_delta']} for x in result['outer_results']])
display(outer)
fig, axes = plt.subplots(1,2,figsize=(12,4))
ridge = candidates[candidates.phase=='ridge']; axes[0].semilogx(ridge.ridge_lambda,ridge.inner_AIA,marker='o'); axes[0].set(xlabel='Fixed Ridge lambda',ylabel='Inner validation AIA (%)',title='SOHO Ridge selection')
axes[1].bar(outer.class_seed.astype(str),outer.delta,color=['tab:green' if x>0 else 'tab:red' for x in outer.delta]); axes[1].axhline(0,color='black',lw=1); axes[1].set(xlabel='Class-order seed',ylabel='SOHO - FLY AIA (pp)',title='Untouched outer validation')
plt.tight_layout(); plt.show()
print('Paired outer summary:', result['outer_paired_soho_minus_fly_aia'])
print('Held-out test authorized:', result['held_out_test_authorized'])


In [ ]:
# Export evidence only. Feature caches are excluded.
archive_base = '/content/soho_imagenetr_optimal_train_only'
archive = shutil.make_archive(archive_base,'zip',root_dir=OUTPUT_DIR)
print('ZIP:', archive, 'bytes=', Path(archive).stat().st_size, 'sha256=', sha(archive))
from google.colab import files
files.download(archive)
print('STOP: return the ZIP for audit. Do not open ImageNet-R test.')
